# Information

Example notebook to show how to use and apply a **random sampling** based on the **som bmu clusters** (or any other integer grid if provided).<br>
Functionality can be used to both:
- Extent the existing positive labels with additional points sampled from specified categories
- Add new negative labels sampled from the specified "background" categories

Options for **selecting** categories::
- Absolute thresholds, values larger than 1: cut-off categories lower/upper the provided threshold
- Quantiles, values between [0, 1]: cut-off categories lower/upper the provided quantile
- Discrete selection: list of categories to keep

Options for **sampling**:
- Sample each class separately based on a fraction of the total number of available points in a cluster
- Sample based on a merged cluster, which is created temporarily and then used to sample the specified number of points

After sampling, the **additional labels** must be **merged** with the existing labels for the respective class (positives/negatives)


# Set up

In [1]:
import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

import numpy as np
import pandas as pd
import rasterio
import gc

from pathlib import Path

# Custom modules
from beak.utilities.io import save_raster

BASE_PATH = files("beak.data")
gc.enable()


**User** inputs<p>
Base data

In [6]:
ROOT_PATH = BASE_PATH / "PROCESSED" / "regional_mama_nico_upmidwest_102008_500"
PATH_LABELS = ROOT_PATH  / "labels" / "TA2_240609_FILTERED_HM9_TA2_MMS.tif"
PATH_BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_102008_RES_500_MAMA_NICO_UPMIDWEST.tif"

SEED = 42

**SOM BMU** results as input for sampling

In [7]:
MODEL_FOLDER = Path(
        "S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/local/experiments/random_point_sampling/03_cma/regional_nico/som/models"
    )

MODEL_CONFIG = "BASELINE_BISON"
MODEL_RUN = "SOM_BASELINE_BISON_F28_X50_Y50_CMAX50_20240619-121410"

CLUSTER_MAP = (MODEL_FOLDER
               / MODEL_CONFIG
               / MODEL_RUN
               / "exports"
               / "GeoTIFF"
               / "BMU_BMU_label_count.tif"
)


# 1: **Create** file list
...

# 2: **Load** data

### Base Raster

In [8]:
# Base raster
base_raster = rasterio.open(PATH_BASE_RASTER)
base_raster_array = base_raster.read(1)
base_raster_array = np.where(base_raster_array == base_raster.nodata, np.nan, base_raster_array)

### Labels

#### Original labels raster

In [9]:
# Labels raster
labels_raster = rasterio.open(PATH_LABELS)
labels_array = labels_raster.read(1)
labels_array = np.where(labels_array == labels_raster.nodata, np.nan, labels_array)

# Check the count of positives and negatives
print(np.unique(labels_array, return_counts=True))

(array([ 0.,  1., nan]), array([2507109,      43, 1977568], dtype=int64))


#### SOM BMU clusters

In [10]:
from beak.utilities.preparation import _get_unique_values_list
# Load cluster map
cluster_raster = rasterio.open(CLUSTER_MAP)
cluster_array = cluster_raster.read(1)
cluster_array = np.where(cluster_array == cluster_raster.nodata, np.nan, cluster_array)

# Check classes and counts
classes, counts = _get_unique_values_list(cluster_array, verbose=1)


Class: 0.0, Count: 2417905
Class: 1.0, Count: 16789
Class: 2.0, Count: 4820
Class: 3.0, Count: 6179
Class: 4.0, Count: 1108
Class: 7.0, Count: 1057


# 3: Sampling

According to our conventions, negative labels have a value of **0** and positive labels have a value of **1**.<p>
Adding **new** labels to the existing ones, we need to **select** which **classes** we want to use for sampling, either for the **positives** or **negatives**.<p>

For the following examples, we will 
- sample from **Class 0** for the **negatives**
- sample from **Classes 1 and higher** for the **positives**

We will **exclude** any **class** here, but could do this using the `exclude_classes` parameter.

The sampling is based on **three steps**:
- **Selection** of relevant classes: mandatory to ensure that only relevant classes are selected
- **Sampling** from these classes
- **Merge** with existing labels:
  - negatives are intended to **replace** the existing negatives
  - positives are intended to be **added** to the existing positives

The **sampling** can be done by both using the **raster's path** (`_sampling_by_selection` or `_sampling_by_threshold`) or using the **cluster map**, which can be an already loaded raster array (`_sampling_select_classes`).

## Negatives

**Selection** of pixels with **Class 0** can be done as follows:
- Threshold via `_sampling_by_treshold` and `type="negatives"` with value **1** (selects all values below)
- Quantile via `sampling_by_threshold` and `type="negatives"` in range [0, 1] (**not** appropriate)
- Discrete selection via `_sampling_by_selection` using the `include` parameter with value **[0]**
- Discrete selection via `_sampling_by_selection` using the `exclude` parameter with all values except **0**

In [11]:
SELECTION = [0]
THRESHOLD = 1
TARGET_NEGATIVES = 0

### Select classes
#### Option A
**Selection** using `_sampling_by_selection`

In [12]:
from beak.utilities.preparation import _sampling_by_selection

# Selection of pixels with class 0
selection_array, selection_settings = _sampling_by_selection(CLUSTER_MAP, include=SELECTION)

# Get unique values
classes, counts = _get_unique_values_list(selection_array, include_nan=True, verbose=1)


Class: 0.0, Count: 2417905
Class: nan, Count: 2066815


#### Option B
**Selection** using `_sampling_by_threshold`

In [13]:
from beak.utilities.preparation import _sampling_by_threshold
selection_array, settings = _sampling_by_threshold(CLUSTER_MAP, type="negatives", threshold=THRESHOLD)

# Get unique values
classes, counts = _get_unique_values_list(selection_array, include_nan=True, verbose=1)


Class: 0.0, Count: 2417905
Class: nan, Count: 2066815


### Sampling

In [14]:
from beak.utilities.preparation import _sampling_select_random_points

# Sampling
new_negatives, neg_dict = _sampling_select_random_points(array=selection_array,           # the prepared array
                                                         selection=SELECTION,             # the classes to sample from
                                                         strategy=[0.5],                  # the fraction to sample (50%)
                                                         target_value=TARGET_NEGATIVES,   # the output value for sampled pixels
                                                         merge_classes=False,             # merge before sampling or not
                                                         min_px=1,                        # minimum number of pixels to sample
                                                         seed=SEED,                       # seed for reproducibility
)

# Get unique values
classes, counts = _get_unique_values_list(new_negatives, include_nan=True, verbose=1)

Class: 0.0, Count: 1208952
Class: nan, Count: 3275768


## Positives

**Selection** of clusters in order for positive sampling can be done as follows:
- Threshold via `_sampling_by_treshold` and `type="positives"` with value **1** (selects all values equal or above)
- Quantile via `sampling_by_threshold` and `type="positives"` (**not** appropriate here)
- Discrete selection via `_sampling_by_selection` using the `include` parameter with value **[1, 2, 3, 4, 7]**
- Discrete selection via `_sampling_by_selection` using the `exclude` parameter with all values except **0**

In [15]:
SELECTION = [1, 2, 3, 4, 7]
EXCLUDE = [0]
THRESHOLD = 1
TARGET_POSITIVES = 1

### Select classes
#### Option A
**Selection** using `_sampling_by_selection`

In [16]:
from beak.utilities.preparation import _sampling_by_selection

# Selection of pixels not in class 0
selection_array, selection_settings = _sampling_by_selection(CLUSTER_MAP, exclude=EXCLUDE)

# Get unique values
classes, counts = _get_unique_values_list(selection_array, include_nan=True, verbose=1)


Class: 1.0, Count: 16789
Class: 2.0, Count: 4820
Class: 3.0, Count: 6179
Class: 4.0, Count: 1108
Class: 7.0, Count: 1057
Class: nan, Count: 4454767


#### Option B
**Selection** using `_sampling_by_threshold`

In [17]:
from beak.utilities.preparation import _sampling_by_threshold
selection_array, settings = _sampling_by_threshold(CLUSTER_MAP, type="positives", threshold=THRESHOLD)

# Get unique values
selected_classes, selected_classes_counts = _get_unique_values_list(selection_array, include_nan=True, verbose=1)


Class: 1.0, Count: 16789
Class: 2.0, Count: 4820
Class: 3.0, Count: 6179
Class: 4.0, Count: 1108
Class: 7.0, Count: 1057
Class: nan, Count: 4454767


### Sampling

In [18]:
from beak.utilities.preparation import _sampling_select_random_points

# Define strategy:
# List with one single value, like [0.5]: 50% of pixels will be sampled from each class
# List with multiple values, like [0.1, 0.25, ...]: with respect to their order, classes will be sampled with different fractions
STRATEGY=[0.1, 0.25, 0.5, 0.5, 0.5]      

# Sampling
add_positives, pos_dict = _sampling_select_random_points(array=selection_array,           # the prepared array
                                                         selection=SELECTION,             # the classes to sample from
                                                         strategy=STRATEGY,               # the fraction to sample
                                                         target_value=TARGET_POSITIVES,   # the output value for sampled pixels
                                                         merge_classes=False,             # merge before sampling or not
                                                         min_px=1,                        # minimum number of pixels to sample
                                                         seed=SEED,                       # seed for reproducibility
)

# Get unique values
classes, counts = _get_unique_values_list(add_positives, include_nan=True, verbose=1)

Class: 1.0, Count: 7056
Class: nan, Count: 4477664


In [19]:
# Check fractions for classes
sum_px = 0
for _class, _count, _fraction in zip(selected_classes, selected_classes_counts, STRATEGY):
  print(f"Class: {_class}, count: {_count}, fraction: {_fraction}, number of pixels: {round(_count * _fraction)}")
  
  sum_px = sum_px + np.sum(_count * _fraction)

print(f"Total number of pixels: {int(round(sum_px))}")

Class: 1.0, count: 16789, fraction: 0.1, number of pixels: 1679
Class: 2.0, count: 4820, fraction: 0.25, number of pixels: 1205
Class: 3.0, count: 6179, fraction: 0.5, number of pixels: 3090
Class: 4.0, count: 1108, fraction: 0.5, number of pixels: 554
Class: 7.0, count: 1057, fraction: 0.5, number of pixels: 528
Total number of pixels: 7056


## Merge labels

In [20]:
# Create new labels array for example purposes and set-back existing negativesTrue)
new_labels_array = np.copy(labels_array)
new_labels_array = np.where(new_labels_array == TARGET_NEGATIVES, np.nan, new_labels_array)

# Add new negative samples to existing labels
new_labels_array = np.where(new_negatives == TARGET_NEGATIVES, TARGET_NEGATIVES, new_labels_array)

# Add new positive samples to existing labels
new_labels_array = np.where(add_positives == TARGET_POSITIVES, TARGET_POSITIVES, new_labels_array)

Save the rasters to check in **GIS**

In [21]:
save_raster(Path("example_labels_original.tif"), labels_array, metadata=labels_raster.meta)
save_raster(Path("example_labels_sampled_positives.tif"), add_positives, metadata=labels_raster.meta)
save_raster(Path("example_labels_sampled_negatives.tif"), new_negatives, metadata=labels_raster.meta)
save_raster(Path("example_labels_sampled_new_labels.tif"), new_labels_array, metadata=labels_raster.meta, overwrite=True)


# 3: **Remove** outliers
Go on with the data preprocessing steps etc. and choose which labels you want to use.